# Mini Project Data Mining: Forecasting Konsumsi Listrik Rumah Tangga

Notebook ini dibuat untuk kategori **Forecasting (Peramalan)** menggunakan dataset `household_power_consumption.csv`.

Framework yang digunakan adalah **CRISP-DM**:
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

## 1. Business Understanding

Penggunaan listrik rumah tangga perlu dianalisis untuk mengetahui pola konsumsi energi dari waktu ke waktu. Dengan melakukan forecasting, konsumsi listrik pada periode berikutnya dapat diperkirakan berdasarkan data historis.

**Tujuan project:**
- Melakukan peramalan nilai `Global_active_power` untuk menit berikutnya.
- Membangun model forecasting sederhana menggunakan data historis konsumsi listrik.
- Mengevaluasi performa model berdasarkan MAE, RMSE, R2 Score, MAPE, dan akurasi forecasting.

**Manfaat analisis:**
- Membantu memahami pola konsumsi listrik rumah tangga.
- Memberikan dasar prediksi kebutuhan energi jangka pendek.
- Membantu pengambilan keputusan terkait efisiensi penggunaan listrik.

## Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import joblib

pd.set_option("display.max_columns", None)

## 2. Data Understanding

Pada tahap ini dataset dibaca, ditampilkan informasi awalnya, serta diperiksa jumlah data, tipe data, missing value, dan data duplikat.

In [ ]:
# Load dataset
DATA_PATH = Path("household_power_consumption.csv")

# Alternatif path jika notebook dijalankan dari folder /mnt/data
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/household_power_consumption.csv")

df = pd.read_csv(DATA_PATH)

# Salinan dataframe dengan nama data agar mudah digunakan di cell berikutnya
data = df.copy()

print("Jumlah data awal:", data.shape)
data.head()

In [ ]:
# Informasi struktur dataset
data.info()

In [ ]:
# Statistik deskriptif awal
data.describe(include="all")

In [ ]:
# Cek missing value dan duplikat
print("Missing value per kolom:")
print(data.isna().sum())

print("\nJumlah data duplikat:", data.duplicated().sum())

## 3. Data Preparation

Tahap ini mencakup:
- Menggabungkan kolom `Date` dan `Time` menjadi `Datetime`.
- Mengubah kolom numerik dari object menjadi numeric.
- Menghapus missing value dan duplikat.
- Membuat fitur lag untuk forecasting.
- Membuat target prediksi yaitu konsumsi listrik pada menit berikutnya.

In [ ]:
# Membuat kolom Datetime
data["Datetime"] = pd.to_datetime(
    data["Date"].astype(str) + " " + data["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

# Konversi kolom numerik
kolom_numerik = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for kolom in kolom_numerik:
    data[kolom] = pd.to_numeric(data[kolom], errors="coerce")

# Cleaning
data = data.dropna()
data = data.drop_duplicates()
data = data.sort_values("Datetime").reset_index(drop=True)

print("Jumlah data setelah cleaning:", data.shape)
data.head()

In [ ]:
# Visualisasi awal: pola Global Active Power pada 1000 data pertama
plot_data = data.head(1000)

plt.figure(figsize=(12, 5))
plt.plot(plot_data["Datetime"], plot_data["Global_active_power"])
plt.title("Pola Global Active Power pada 1000 Data Pertama")
plt.xlabel("Waktu")
plt.ylabel("Global Active Power")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Visualisasi distribusi konsumsi daya aktif
plt.figure(figsize=(8, 5))
plt.hist(data["Global_active_power"], bins=50)
plt.title("Distribusi Global Active Power")
plt.xlabel("Global Active Power")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

In [ ]:
# Menggunakan sebagian data agar proses training cepat saat dijalankan di laptop
# Data tetap memenuhi syarat karena jumlahnya jauh di atas 5000 data.
data_model = data.head(200000).copy()

# Membuat fitur waktu
data_model["hour"] = data_model["Datetime"].dt.hour
data_model["dayofweek"] = data_model["Datetime"].dt.dayofweek
data_model["month"] = data_model["Datetime"].dt.month

# Membuat fitur lag dari Global_active_power
for lag in [1, 2, 3, 5, 10]:
    data_model[f"gap_lag_{lag}"] = data_model["Global_active_power"].shift(lag)

# Target forecasting: Global_active_power pada menit berikutnya
data_model["target_next_minute"] = data_model["Global_active_power"].shift(-1)

# Hapus baris kosong akibat pembuatan lag dan target
data_model = data_model.dropna().reset_index(drop=True)

print("Jumlah data untuk modeling:", data_model.shape)
data_model.head()

## 4. Modeling

Algoritma yang digunakan adalah **Linear Regression**.  
Model dilatih untuk memprediksi `Global_active_power` pada menit berikutnya menggunakan data historis dan fitur waktu.

In [ ]:
# Menentukan fitur dan target
fitur = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "gap_lag_1",
    "gap_lag_2",
    "gap_lag_3",
    "gap_lag_5",
    "gap_lag_10",
    "hour",
    "dayofweek",
    "month"
]

X = data_model[fitur]
y = data_model["target_next_minute"]

# Split data secara urutan waktu: 80% training, 20% testing
split_index = int(len(data_model) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Jumlah data training:", X_train.shape)
print("Jumlah data testing:", X_test.shape)

# Training model
model = LinearRegression()
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)

## 5. Evaluation

Evaluasi dilakukan menggunakan:
- MAE
- RMSE
- R2 Score
- MAPE
- Akurasi Forecasting = `(1 - MAPE) x 100%`

Pada project forecasting ini, kriteria sukses dianggap tercapai jika akurasi forecasting minimal 80%.

In [ ]:
# Evaluasi model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
akurasi_forecasting = max(0, (1 - mape) * 100)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)
print("MAPE:", mape)
print("Akurasi Forecasting (%):", akurasi_forecasting)

if akurasi_forecasting >= 80:
    print("Kesimpulan Evaluasi: Model memenuhi kriteria sukses minimal 80%.")
else:
    print("Kesimpulan Evaluasi: Model belum memenuhi kriteria sukses minimal 80%.")

In [ ]:
# Tabel hasil prediksi
hasil_forecasting = pd.DataFrame({
    "Datetime": data_model["Datetime"].iloc[split_index:].values,
    "Aktual": y_test.values,
    "Prediksi": y_pred
})

hasil_forecasting.head(20)

In [ ]:
# Visualisasi hasil forecasting
plt.figure(figsize=(12, 5))
plt.plot(hasil_forecasting["Datetime"].head(300), hasil_forecasting["Aktual"].head(300), label="Aktual")
plt.plot(hasil_forecasting["Datetime"].head(300), hasil_forecasting["Prediksi"].head(300), label="Prediksi")
plt.title("Perbandingan Nilai Aktual dan Prediksi Global Active Power")
plt.xlabel("Waktu")
plt.ylabel("Global Active Power")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Deployment

Pada tahap deployment, hasil project disiapkan agar dapat dipublikasikan di Google Sites dan GitHub.  
Output yang disimpan:
- File hasil prediksi `forecasting_results.csv`
- Model forecasting `forecasting_model.pkl`

File tersebut dapat digunakan sebagai bukti hasil analisis dan pendukung dokumentasi project.

In [ ]:
# Simpan hasil forecasting dan model
hasil_forecasting.to_csv("forecasting_results.csv", index=False)
joblib.dump(model, "forecasting_model.pkl")

print("File forecasting_results.csv berhasil disimpan.")
print("File forecasting_model.pkl berhasil disimpan.")

## Kesimpulan

Project forecasting ini berhasil membuat model peramalan konsumsi listrik rumah tangga untuk memprediksi `Global_active_power` pada menit berikutnya. Model dievaluasi menggunakan beberapa metrik, terutama akurasi forecasting. Hasil evaluasi dapat digunakan untuk menentukan apakah model sudah memenuhi kriteria sukses minimal 80%.